# Figure 7 Intuition Scaffold (Toy 2D Example)

This notebook builds an end-to-end pedagogical scaffold for **Figure 7 style pointwise divergence contributions** using a synthetic 2D toy setup.

We will mirror the plotting progression:
1. Plot 29: raw pooled samples
2. Plot 30: shared-bin histograms
3. Plot 31: KDE overlays + signed density difference
4. Plot 32: local (pointwise) KL terms
5. Plot 33: cumulative KL area
6. Final: Figure-7-like pointwise KL / reverse-KL / Jeffreys panel


## Setup

- Synthetic data only (fixed seed).
- Cluster A is the AND reference distribution proxy.
- Cluster B is a source closer to AND.
- Cluster C is a source farther from AND.
- Figures are saved to `tutorials/figures/figure7_intuition_scaffold/`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from scipy.spatial.distance import pdist, cdist

plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=4, suppress=True)

SEED = 7
rng = np.random.default_rng(SEED)

OUTPUT_DIR = Path("tutorials/figures/figure7_intuition_scaffold")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving figures to: {OUTPUT_DIR.resolve()}")


## 1) Generate 3 clusters in 2D (10 points each)


In [ ]:
n_per_cluster = 10

mean_and = np.array([0.0, 0.0])
cov_and = np.array([[0.10, 0.03], [0.03, 0.08]])

mean_src1 = np.array([0.75, 0.60])
cov_src1 = np.array([[0.12, -0.01], [-0.01, 0.10]])

mean_src2 = np.array([2.70, 2.30])
cov_src2 = np.array([[0.14, 0.02], [0.02, 0.12]])

X_and = rng.multivariate_normal(mean_and, cov_and, size=n_per_cluster)
X_src1 = rng.multivariate_normal(mean_src1, cov_src1, size=n_per_cluster)
X_src2 = rng.multivariate_normal(mean_src2, cov_src2, size=n_per_cluster)

cluster_meta = {
    "AND": {
        "data": X_and,
        "color": "#1f77b4",
        "label": "Cluster A: AND reference",
    },
    "SRC1": {
        "data": X_src1,
        "color": "#2ca02c",
        "label": "Cluster B: Source-1 (closer)",
    },
    "SRC2": {
        "data": X_src2,
        "color": "#d62728",
        "label": "Cluster C: Source-2 (farther)",
    },
}

fig, ax = plt.subplots(figsize=(7.2, 6.2))
for key, meta in cluster_meta.items():
    pts = meta["data"]
    ax.scatter(
        pts[:, 0],
        pts[:, 1],
        s=70,
        color=meta["color"],
        alpha=0.90,
        edgecolor="white",
        linewidth=0.7,
        label=meta["label"],
    )
    for idx, (x, y) in enumerate(pts):
        ax.text(x + 0.03, y + 0.03, str(idx), fontsize=8, color=meta["color"], alpha=0.9)

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Toy 2D data: three clusters (10 points each)")
ax.legend(loc="upper left", frameon=True)
ax.set_aspect("equal", adjustable="box")

fig_path = OUTPUT_DIR / "plot_00_toy_2d_clusters.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")


### Interpretation (2D foundation)

- We intentionally created one **reference cluster** (AND) and two sources with different proximity.
- The closer source should produce smaller residual mismatch to AND, while the farther source should produce larger mismatch.
- The pointwise divergence plots later are built from a **1D distance axis** derived from these 2D points.


## 2) Build the 1D terminal-distance axis $d_T$

We define distributions along a scalar axis:- $P_{\mathrm{AND}}$: **within-AND variability** (pairwise distances among AND points).- $P_{p^*}$: **source-vs-AND residual distances** (all cross distances from source points to AND points).

This mirrors the Figure 7 interpretation: intrinsic AND spread vs source residual mismatch.


In [ ]:
d_within_and = pdist(X_and, metric="euclidean")
d_src1_vs_and = cdist(X_src1, X_and, metric="euclidean").ravel()
d_src2_vs_and = cdist(X_src2, X_and, metric="euclidean").ravel()

distance_samples = {
    "within_and": d_within_and,
    "source_1": d_src1_vs_and,
    "source_2": d_src2_vs_and,
}

dist_meta = {
    "source_1": {"label": "Source-1 (closer)", "color": "#2ca02c"},
    "source_2": {"label": "Source-2 (farther)", "color": "#d62728"},
}

summary_df = pd.DataFrame(
    {
        "distribution": ["within_and", "source_1", "source_2"],
        "n_samples": [len(d_within_and), len(d_src1_vs_and), len(d_src2_vs_and)],
        "mean": [d_within_and.mean(), d_src1_vs_and.mean(), d_src2_vs_and.mean()],
        "std": [d_within_and.std(ddof=1), d_src1_vs_and.std(ddof=1), d_src2_vs_and.std(ddof=1)],
        "min": [d_within_and.min(), d_src1_vs_and.min(), d_src2_vs_and.min()],
        "max": [d_within_and.max(), d_src1_vs_and.max(), d_src2_vs_and.max()],
    }
)
summary_df


## 3) Shared grid, KDE PMFs, and contribution formulas

We estimate PMFs on a common grid (uniform spacing $\Delta x$).

$$
k_{\mathrm{AND}	o p^*}(x_i)=\dfrac{p_{\mathrm{AND}}(i)\log\left(\dfrac{p_{\mathrm{AND}}(i)}{p_{p^*}(i)}
\right)}{\Delta x}
$$

$$
k_{p^*	o\mathrm{AND}}(x_i)=\dfrac{p_{p^*}(i)\log\left(\dfrac{p_{p^*}(i)}{p_{\mathrm{AND}}(i)}
\right)}{\Delta x}
$$

$$
j(x_i)=\dfrac{\left(p_{\mathrm{AND}}(i)-p_{p^*}(i)
\right)\log\left(\dfrac{p_{\mathrm{AND}}(i)}{p_{p^*}(i)}
\right)}{\Delta x}
$$

Areas/integrals:

$$
\int k_{\mathrm{AND}	o p^*}(x)\,dx = D_{KL}(P_{\mathrm{AND}}\|P_{p^*})
$$

$$
\int k_{p^*	o\mathrm{AND}}(x)\,dx = D_{KL}(P_{p^*}\|P_{\mathrm{AND}})
$$

$$
\int j(x)\,dx = J(P_{\mathrm{AND}},P_{p^*}) = D_{KL}(P_{\mathrm{AND}}\|P_{p^*}) + D_{KL}(P_{p^*}\|P_{\mathrm{AND}})
$$


In [ ]:
EPS = 1e-12
all_vals = np.concatenate(list(distance_samples.values()))

x_min = max(all_vals.min() * 0.80, 0.0)
x_max = all_vals.max() * 1.15
n_grid = 1200
x_grid = np.linspace(x_min, x_max, n_grid)
dx = float(x_grid[1] - x_grid[0])


def kde_to_pmf(samples: np.ndarray, x_grid: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    kde = stats.gaussian_kde(samples, bw_method="scott")
    density = np.clip(kde(x_grid), eps, None)
    pmf = density * dx
    pmf = pmf / pmf.sum()
    return pmf


def local_kl_density(p: np.ndarray, q: np.ndarray, dx: float):
    contrib = p * np.log(p / q)
    return contrib / dx, float(np.sum(contrib))


def jeffreys_density(p: np.ndarray, q: np.ndarray, dx: float):
    contrib = (p - q) * np.log(p / q)
    return contrib / dx, float(np.sum(contrib))


def plot_signed_fill(ax, x, y, color):
    ax.fill_between(x, 0.0, y, where=(y >= 0), color=color, alpha=0.30, linewidth=0)
    ax.fill_between(x, 0.0, y, where=(y < 0), color="#666666", alpha=0.18, linewidth=0)
    ax.plot(x, y, color=color, lw=2.0)
    ax.axhline(0.0, color="#666666", lw=0.9, alpha=0.8)


def overlay_kdes(ax, x, p_and, p_src, dx, and_color="#1f77b4", src_color="#2ca02c"):
    d_and = p_and / dx
    d_src = p_src / dx
    ax.plot(x, d_and, ls="--", lw=1.2, color=and_color, alpha=0.45, zorder=1)
    ax.plot(x, d_src, ls="--", lw=1.2, color=src_color, alpha=0.45, zorder=1)


def save_and_show(fig, filename: str, dpi: int = 180):
    out_path = OUTPUT_DIR / filename
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")


p_and = kde_to_pmf(distance_samples["within_and"], x_grid, EPS)
p_sources = {
    "source_1": kde_to_pmf(distance_samples["source_1"], x_grid, EPS),
    "source_2": kde_to_pmf(distance_samples["source_2"], x_grid, EPS),
}

density_and = p_and / dx
density_sources = {k: v / dx for k, v in p_sources.items()}

print(f"Shared grid: n_grid={n_grid}, dx={dx:.6f}, x-range=[{x_min:.4f}, {x_max:.4f}]")


In [ ]:
print("PMF normalization checks")
print(f"  sum(P_AND) = {p_and.sum():.8f}")
for src in ["source_1", "source_2"]:
    p_src = p_sources[src]
    k_fwd_y, kl_fwd = local_kl_density(p_and, p_src, dx)
    k_rev_y, kl_rev = local_kl_density(p_src, p_and, dx)
    j_y, j_val = jeffreys_density(p_and, p_src, dx)
    # Numerical integral checks on density form
    kl_fwd_integral = np.sum(k_fwd_y) * dx
    kl_rev_integral = np.sum(k_rev_y) * dx
    j_integral = np.sum(j_y) * dx
    print(f"\n{dist_meta[src]['label']}")
    print(f"  sum(P_src)              = {p_src.sum():.8f}")
    print(f"  KL(AND||src)            = {kl_fwd:.8f}")
    print(f"  Integral[k_AND->src]    = {kl_fwd_integral:.8f}")
    print(f"  KL(src||AND)            = {kl_rev:.8f}")
    print(f"  Integral[k_src->AND]    = {kl_rev_integral:.8f}")
    print(f"  Jeffreys                = {j_val:.8f}")
    print(f"  Integral[j(x)]          = {j_integral:.8f}")
    print(f"  KL_fwd + KL_rev         = {(kl_fwd + kl_rev):.8f}")
    assert np.isclose(p_src.sum(), 1.0, atol=1e-6)
    assert np.isclose(kl_fwd, kl_fwd_integral, atol=1e-8)
    assert np.isclose(kl_rev, kl_rev_integral, atol=1e-8)
    assert np.isclose(j_val, j_integral, atol=1e-8)
    assert np.isclose(j_val, kl_fwd + kl_rev, atol=1e-8)
print("\nAll sanity checks passed.")


### Sanity check interpretation

- PMFs are normalized on the shared grid (`sum ≈ 1`).
- Integrating each local density recovers the corresponding scalar divergence.
- Jeffreys is numerically equal to forward-KL + reverse-KL, as expected.


## 4) Plot 29: raw pooled samples


### Plot 29 Expanded: How the first \(d_T\) values are computed

For this notebook:
- `within_AND` comes from `pdist(X_and)` (pairwise distances inside AND cluster).
- `source_1` comes from `cdist(X_src1, X_and)` (cross distances from source-1 points to AND points).

Distance formula in both cases:
\[
d((x_1,y_1),(x_2,y_2)) = \sqrt{(x_1-x_2)^2 + (y_1-y_2)^2}
\]

Below we compute just a few explicit examples (not the full matrix).


In [ ]:
example_within_pairs = [(0, 1), (0, 2), (3, 7)]
example_cross_pairs = [(0, 0), (1, 4), (6, 2)]
within_rows = []
for i, j in example_within_pairs:
    p_i, p_j = X_and[i], X_and[j]
    dx_ = p_i[0] - p_j[0]
    dy_ = p_i[1] - p_j[1]
    dist = float(np.sqrt(dx_**2 + dy_**2))
    within_rows.append(
        {
            "pair": f"AND[{i}] vs AND[{j}]",
            "x_diff": dx_,
            "y_diff": dy_,
            "distance": dist,
        }
    )
within_examples_df = pd.DataFrame(within_rows)
cross_rows = []
for s_idx, a_idx in example_cross_pairs:
    p_s, p_a = X_src1[s_idx], X_and[a_idx]
    dx_ = p_s[0] - p_a[0]
    dy_ = p_s[1] - p_a[1]
    dist = float(np.sqrt(dx_**2 + dy_**2))
    cross_rows.append(
        {
            "pair": f"SRC1[{s_idx}] vs AND[{a_idx}]",
            "x_diff": dx_,
            "y_diff": dy_,
            "distance": dist,
        }
    )
cross_examples_df = pd.DataFrame(cross_rows)
print("Manual examples for within_AND = pdist(X_and):")
print(within_examples_df.to_string(index=False))
print("\nManual examples for source_1 = cdist(X_src1, X_and):")
print(cross_examples_df.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4), sharey=True)

# Left panel: within_AND selected pdist examples
ax = axes[0]
ax.scatter(X_and[:, 0], X_and[:, 1], color="#1f77b4", s=65, alpha=0.9, edgecolor="white", linewidth=0.6)
for i, j in example_within_pairs:
    p_i, p_j = X_and[i], X_and[j]
    d_ij = np.linalg.norm(p_i - p_j)
    ax.plot([p_i[0], p_j[0]], [p_i[1], p_j[1]], color="#1f77b4", lw=2.1)
    mid = 0.5 * (p_i + p_j)
    ax.text(mid[0], mid[1], f"{d_ij:.2f}", color="#1f77b4", fontsize=9, bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"))
for idx in sorted({i for pair in example_within_pairs for i in pair}):
    ax.text(X_and[idx, 0] + 0.02, X_and[idx, 1] + 0.02, f"A{idx}", fontsize=8, color="#1f77b4")
ax.set_title("Selected within_AND distances (pdist)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal", adjustable="box")

# Right panel: source_1 selected cdist examples
ax = axes[1]
ax.scatter(X_and[:, 0], X_and[:, 1], color="#1f77b4", s=45, alpha=0.35, edgecolor="white", linewidth=0.4, label="AND points")
ax.scatter(X_src1[:, 0], X_src1[:, 1], color="#2ca02c", s=65, alpha=0.9, edgecolor="white", linewidth=0.6, label="Source-1 points")
for s_idx, a_idx in example_cross_pairs:
    p_s, p_a = X_src1[s_idx], X_and[a_idx]
    d_sa = np.linalg.norm(p_s - p_a)
    ax.plot([p_s[0], p_a[0]], [p_s[1], p_a[1]], color="#2ca02c", lw=2.1, ls="--")
    mid = 0.5 * (p_s + p_a)
    ax.text(mid[0], mid[1], f"{d_sa:.2f}", color="#2ca02c", fontsize=9, bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"))
for s_idx, a_idx in example_cross_pairs:
    ax.text(X_src1[s_idx, 0] + 0.02, X_src1[s_idx, 1] + 0.02, f"S{s_idx}", fontsize=8, color="#2ca02c")
    ax.text(X_and[a_idx, 0] + 0.02, X_and[a_idx, 1] + 0.02, f"A{a_idx}", fontsize=8, color="#1f77b4")
ax.set_title("Selected source_1 distances (cdist)")
ax.set_xlabel("x")
ax.legend(loc="upper left", fontsize=8, frameon=True)
ax.set_aspect("equal", adjustable="box")

fig.suptitle("A few explicit distance calculations used in Plot 29", y=1.01)
fig.tight_layout()
save_and_show(fig, "plot_29a_distance_examples_geometry.png")


### Demonstrate how \(d_T\) is partitioned into shared bins

Plot 29 itself shows raw samples, but the next steps (plots 30+) require shared bins/grid regions.
To make that concrete, we place a few manually computed distances into explicit intervals.


In [ ]:
selected_distances = np.array(list(within_examples_df["distance"]) + list(cross_examples_df["distance"]))
selected_labels = list(within_examples_df["pair"]) + list(cross_examples_df["pair"])

bins = np.linspace(0.0, max(d_within_and.max(), d_src1_vs_and.max()) * 1.02, 9)
raw_bin_idx = np.digitize(selected_distances, bins, right=False) - 1
bin_idx = np.clip(raw_bin_idx, 0, len(bins) - 2)

partition_df = pd.DataFrame(
    {
        "example": selected_labels,
        "d_T": selected_distances,
        "bin_index": bin_idx,
        "bin_interval": [f"[{bins[b]:.2f}, {bins[b+1]:.2f})" for b in bin_idx],
    }
).sort_values("d_T")

print("Selected distance values and their shared-bin assignments:")
print(partition_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(11.6, 4.9))
ax.hist(d_within_and, bins=bins, density=True, color="#1f77b4", alpha=0.28, edgecolor="white", linewidth=0.5, label="within_AND (pdist)")
ax.hist(d_src1_vs_and, bins=bins, density=True, color="#2ca02c", alpha=0.28, edgecolor="white", linewidth=0.5, label="source_1 (cdist)")

for edge in bins:
    ax.axvline(edge, color="#888888", lw=0.8, alpha=0.35)

y_top = ax.get_ylim()[1]
y_mark = 0.05 * y_top
for lbl, x in zip(selected_labels, selected_distances):
    ax.scatter([x], [y_mark], color="black", s=16, zorder=5)
    ax.text(x, y_mark * 1.16, lbl, rotation=35, fontsize=7.5, ha="left", va="bottom")

ax.set_xlabel(r"$d_T$")
ax.set_ylabel("Density")
ax.set_title("Shared-bin partitioning of selected $d_T$ examples")
ax.legend(loc="upper right", fontsize=9, frameon=False)
ax.grid(alpha=0.20)

fig.tight_layout()
save_and_show(fig, "plot_29b_dt_partition_demo.png")


### Why this matters for Plot 29

- Plot 29 is the **raw measurement layer**: each dot is one computed distance.
- The worked examples above show exactly where those dots come from (`pdist` vs `cdist`).
- Partitioning \(d_T\) into shared intervals is the bridge to plots 30-33, where local mass and divergence contributions are computed.


In [ ]:
series = [
    ("Within-AND", distance_samples["within_and"], "#1f77b4"),
    (dist_meta["source_1"]["label"], distance_samples["source_1"], dist_meta["source_1"]["color"]),
    (dist_meta["source_2"]["label"], distance_samples["source_2"], dist_meta["source_2"]["color"]),
]

y_pos = np.arange(len(series))
jitter_rng = np.random.default_rng(42)

fig, ax = plt.subplots(figsize=(11, 5.2))
for yi, (label, vals, color) in enumerate(series):
    jitter = jitter_rng.uniform(-0.12, 0.12, size=len(vals))
    ax.scatter(vals, yi + jitter, s=24, alpha=0.28, color=color, edgecolors="white", linewidths=0.5)
    m = vals.mean()
    ax.plot([m, m], [yi - 0.24, yi + 0.24], color=color, lw=3.2)
    ax.text(m, yi + 0.30, f"mean={m:.3f}", color=color, fontsize=9, ha="center")

ax.set_yticks(y_pos)
ax.set_yticklabels([s[0] for s in series])
ax.set_xlabel(r"$d_T$ (terminal distance axis)")
ax.set_title("Plot 29 — Raw pooled $d_T$ samples (pointwise scaffold 1/5)")
ax.grid(alpha=0.22)

save_and_show(fig, "plot_29_pointwise_scaffold_raw_samples.png")


### How to read Plot 29

- Each dot is a raw \(d_T\) sample before any smoothing.
- The vertical tick is the sample mean for that distribution.
- This view shows where overlap/mismatch begins, but not yet *local mass ratios* needed for KL.


## 5) Plot 30: shared-bin histograms


In [ ]:
sources = ["source_1", "source_2"]
bins = np.linspace(x_grid.min(), x_grid.max(), 46)

fig, axes = plt.subplots(len(sources), 1, figsize=(11.5, 6.2), sharex=True, squeeze=False)
for ridx, src in enumerate(sources):
    ax = axes[ridx][0]
    src_label = dist_meta[src]["label"]
    src_color = dist_meta[src]["color"]

    ax.hist(
        distance_samples["within_and"],
        bins=bins,
        density=True,
        color="#1f77b4",
        alpha=0.30,
        edgecolor="white",
        linewidth=0.5,
        label=r"Within-AND proxy $P_{AND}$",
    )
    ax.hist(
        distance_samples[src],
        bins=bins,
        density=True,
        color=src_color,
        alpha=0.30,
        edgecolor="white",
        linewidth=0.5,
        label=rf"{src_label} $P_{{p^*}}$",
    )

    ax.set_ylabel("Density")
    ax.set_title(f"{src_label}: shared bins for direct local-mass comparison", fontsize=10)
    ax.legend(loc="upper right", frameon=False, fontsize=9)
    ax.grid(alpha=0.20)

axes[-1][0].set_xlabel(r"$d_T$")
fig.suptitle("Plot 30 — Shared-bin histograms (pointwise scaffold 2/5)", y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.96))

save_and_show(fig, "plot_30_pointwise_scaffold_hist_shared_bins.png")


### How to read Plot 30

- Both distributions use the same bins, so each bin compares the same \(d_T\) region.
- Histogram overlap suggests agreement; separated bars suggest local mismatch.
- This is the discrete-mass intuition before smooth KDE-based pointwise terms.


## 6) Plot 31: KDE overlays + signed density difference


In [ ]:
fig, axes = plt.subplots(len(sources), 1, figsize=(11.5, 6.4), sharex=True, squeeze=False)

for ridx, src in enumerate(sources):
    ax = axes[ridx][0]
    src_label = dist_meta[src]["label"]
    src_color = dist_meta[src]["color"]

    d_src = density_sources[src]
    diff = density_and - d_src

    ax.plot(x_grid, density_and, ls="--", lw=2.0, color="#1f77b4", label=r"KDE $P_{AND}$")
    ax.plot(x_grid, d_src, ls="--", lw=2.0, color=src_color, label=rf"KDE {src_label}")

    ax.fill_between(x_grid, 0.0, diff, where=(diff >= 0), color="#1f77b4", alpha=0.14, linewidth=0)
    ax.fill_between(x_grid, 0.0, diff, where=(diff < 0), color=src_color, alpha=0.14, linewidth=0)
    ax.axhline(0.0, color="#777777", lw=0.9, alpha=0.75)

    ax.set_ylabel("Density")
    ax.set_title(f"{src_label}: signed density difference ($P_{{AND}} - P_{{p^*}}$)", fontsize=10)
    ax.legend(loc="upper right", frameon=False, fontsize=9)
    ax.grid(alpha=0.20)

axes[-1][0].set_xlabel(r"$d_T$")
fig.suptitle("Plot 31 — KDE overlays + signed difference (pointwise scaffold 3/5)", y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.96))

save_and_show(fig, "plot_31_pointwise_scaffold_kde_overlay_diff.png")
r

### How to read Plot 31

- Dashed curves are smooth density estimates on a shared axis.
- Blue-filled regions indicate \(P_{AND}\) locally exceeds \(P_{p^*}\); source-colored regions indicate the opposite.
- These local differences are exactly what KL weights asymmetrically through \(\log(p/q)\).


## 7) Plot 32: local (pointwise) KL contribution terms


### NEW (March 4): Real-data bridge (Plot 31 -> Plot 32)

This block uses the **same real distributions** from earlier cells:
- `P_AND` = `distance_samples["within_and"]`
- `Q` = `distance_samples["source_1"]` or `distance_samples["source_2"]`

For each source row:
1. Left panel: shared-bin masses (`p_i`, `q_i`).
2. Right panel: local KL bars computed from those exact masses:
   - `k_AND->src(i) = p_i * log(p_i/q_i)`
   - `k_src->AND(i) = q_i * log(q_i/p_i)`

So this is the explicit jump from Plot 30/31 intuition to Plot 32 mechanics.


In [ ]:
bridge_bins = np.linspace(x_grid.min(), x_grid.max(), 24)
bridge_centers = 0.5 * (bridge_bins[:-1] + bridge_bins[1:])
bar_w = (bridge_bins[1] - bridge_bins[0]) * 0.42
eps_b = 1e-12

fig, axes = plt.subplots(len(sources), 2, figsize=(14.6, 8.2), sharex=True, squeeze=False)

for ridx, src in enumerate(sources):
    src_label = dist_meta[src]["label"]
    src_color = dist_meta[src]["color"]

    p_vals = distance_samples["within_and"]
    q_vals = distance_samples[src]

    c_p, _ = np.histogram(p_vals, bins=bridge_bins)
    c_q, _ = np.histogram(q_vals, bins=bridge_bins)

    p_bin = c_p.astype(float)
    q_bin = c_q.astype(float)
    p_bin = p_bin / p_bin.sum()
    q_bin = q_bin / q_bin.sum()

    k_fwd = p_bin * np.log((p_bin + eps_b) / (q_bin + eps_b))
    k_rev = q_bin * np.log((q_bin + eps_b) / (p_bin + eps_b))

    # Left: same local masses (pre-KL view)
    axL = axes[ridx, 0]
    axL.bar(bridge_centers - bar_w / 2, p_bin, width=bar_w, color="#1f77b4", alpha=0.80, label="P_AND bin mass")
    axL.bar(bridge_centers + bar_w / 2, q_bin, width=bar_w, color=src_color, alpha=0.66, label=f"{src_label} bin mass")
    axL.set_ylabel("Probability mass")
    axL.set_title(f"{src_label}: shared-bin masses")
    axL.grid(alpha=0.20)
    axL.legend(loc="upper right", frameon=False, fontsize=8)

    # Right: local KL from those same bins
    axR = axes[ridx, 1]
    col_fwd = [src_color if v >= 0 else "#666666" for v in k_fwd]
    col_rev = [src_color if v >= 0 else "#A0A0A0" for v in k_rev]

    axR.bar(
        bridge_centers - bar_w / 2,
        k_fwd,
        width=bar_w,
        color=col_fwd,
        alpha=0.85,
        label=f"k_AND->src (sum={k_fwd.sum():.3f})",
    )
    axR.bar(
        bridge_centers + bar_w / 2,
        k_rev,
        width=bar_w,
        color=col_rev,
        alpha=0.62,
        label=f"k_src->AND (sum={k_rev.sum():.3f})",
    )
    axR.axhline(0.0, color="#666666", lw=0.9)
    axR.set_ylabel("Local KL contribution")
    axR.set_title(f"{src_label}: local KL from same bins")
    axR.grid(alpha=0.20)
    axR.legend(loc="upper right", frameon=False, fontsize=8)

# print a few bins to make the mapping explicit
src = "source_1"
c_p, _ = np.histogram(distance_samples["within_and"], bins=bridge_bins)
c_q, _ = np.histogram(distance_samples[src], bins=bridge_bins)
p_bin = c_p.astype(float); q_bin = c_q.astype(float)
p_bin = p_bin / p_bin.sum(); q_bin = q_bin / q_bin.sum()
k_fwd = p_bin * np.log((p_bin + eps_b) / (q_bin + eps_b))

bridge_df = pd.DataFrame({
    "bin_center": bridge_centers,
    "p_i(P_AND)": p_bin,
    "q_i(source_1)": q_bin,
    "k_i=p_i*log(p_i/q_i)": k_fwd,
})
print("First 10 shared bins (source_1) showing mass -> local KL mapping:")
print(bridge_df.head(10).round(6).to_string(index=False))

for c in range(2):
    axes[-1, c].set_xlabel("d_T (shared bin centers)")

fig.suptitle("NEW bridge: same binned masses transformed into Plot-32 local KL bars", y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.97))
save_and_show(fig, "plot_31_to_32_bridge_real_data_new.png")


**Answer to your specific question:**
- Yes, `Q_close` and `Q_tail_miss` are two different **toy** Q distributions used only for intuition.
- In your real pipeline, Q is `source_1` or `source_2`.
- In the bridge figure, the top bars and bottom bars come from the **same bins**; the bottom is just `p_i,q_i` passed through the KL formula.


### Step-by-step: local KL from shared-bin masses (real data)

This cell shows the exact mechanics for one real source (`source_1`) using shared bins.

1. Use the **same bin edges** for AND and source.
2. Convert histogram counts to masses: `p_i` and `q_i`.
3. Compute local terms per bin:
   - `k_AND->src(i) = p_i * log(p_i / q_i)`
   - `k_src->AND(i) = q_i * log(q_i / p_i)`
4. Sum over bins to recover KL.

Why spikes happen:
- A bin spikes when one distribution has nontrivial mass and the other has near-zero mass.
- The `log(p_i/q_i)` factor grows quickly when `q_i` is tiny, so that bin contributes a lot.


In [ ]:
source_demo = "source_1"
source_demo_label = dist_meta[source_demo]["label"]
source_demo_color = dist_meta[source_demo]["color"]
shared_bins_demo = np.linspace(x_grid.min(), x_grid.max(), 24)
bin_centers_demo = 0.5 * (shared_bins_demo[:-1] + shared_bins_demo[1:])
bin_w_demo = shared_bins_demo[1] - shared_bins_demo[0]
eps_demo = 1e-12
# 1) Same bins for both distributions
counts_and, _ = np.histogram(distance_samples["within_and"], bins=shared_bins_demo)
counts_src, _ = np.histogram(distance_samples[source_demo], bins=shared_bins_demo)
# 2) Normalize to local masses
p_i = counts_and.astype(float)
q_i = counts_src.astype(float)
p_i = p_i / p_i.sum()
q_i = q_i / q_i.sum()
# 3) Local KL terms
k_and_src = p_i * np.log((p_i + eps_demo) / (q_i + eps_demo))
k_src_and = q_i * np.log((q_i + eps_demo) / (p_i + eps_demo))
# Build table
rows = []
for i in range(len(bin_centers_demo)):
    rows.append(
        {
            "bin_idx": i,
            "bin_interval": f"[{shared_bins_demo[i]:.3f}, {shared_bins_demo[i+1]:.3f})",
            "p_i": p_i[i],
            "q_i": q_i[i],
            "k_AND_to_src": k_and_src[i],
            "k_src_to_AND": k_src_and[i],
            "abs_k_AND_to_src": abs(k_and_src[i]),
        }
    )
df_local = pd.DataFrame(rows)
df_nonzero = df_local[(df_local["p_i"] > 0) | (df_local["q_i"] > 0)].copy()
df_top = df_nonzero.sort_values("abs_k_AND_to_src", ascending=False).head(8)
print(f"Local KL demo source: {source_demo_label}")
print(f"KL(P_AND || Q) = {k_and_src.sum():.6f}")
print(f"KL(Q || P_AND) = {k_src_and.sum():.6f}")
# One explicit worked bin
b = int(df_top.iloc[0]["bin_idx"])
pb = p_i[b]
qb = q_i[b]
kb = k_and_src[b]
print("\nWorked single-bin example (largest |k_AND->src| bin):")
print(f"  bin {b} interval [{shared_bins_demo[b]:.3f}, {shared_bins_demo[b+1]:.3f})")
print(f"  p_i = {pb:.6f}, q_i = {qb:.6f}")
print(f"  k_AND->src = p_i*log(p_i/q_i) = {pb:.6f} * log({pb:.6f}/{qb:.6f}) = {kb:.6f}")
print("\nTop 8 bins by |k_AND->src|:")
print(
    df_top[
        ["bin_idx", "bin_interval", "p_i", "q_i", "k_AND_to_src", "k_src_to_AND"]
    ].round(6).to_string(index=False)
)
# Plot: same bins (mass) -> local KL on the same bins
sel_idx = np.sort(df_top["bin_idx"].astype(int).values)
labels = [f"[{shared_bins_demo[i]:.2f},{shared_bins_demo[i+1]:.2f})" for i in sel_idx]
x = np.arange(len(sel_idx))
w = 0.38
fig, axes = plt.subplots(1, 2, figsize=(15.2, 5.6))
# Left: masses from shared bins
ax = axes[0]
ax.bar(x - w/2, p_i[sel_idx], width=w, color="#1f77b4", alpha=0.82, label="p_i = P_AND bin mass")
ax.bar(x + w/2, q_i[sel_idx], width=w, color=source_demo_color, alpha=0.70, label=f"q_i = {source_demo_label} bin mass")
ax.set_title("Same bins: local masses")
ax.set_ylabel("Mass per bin")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
ax.grid(alpha=0.20)
ax.legend(frameon=False, fontsize=8)
# Right: local KL from those same masses
ax = axes[1]
col_fwd = [source_demo_color if v >= 0 else "#666666" for v in k_and_src[sel_idx]]
col_rev = [source_demo_color if v >= 0 else "#A0A0A0" for v in k_src_and[sel_idx]]
ax.bar(x - w/2, k_and_src[sel_idx], width=w, color=col_fwd, alpha=0.86, label="k_AND->src(i)")
ax.bar(x + w/2, k_src_and[sel_idx], width=w, color=col_rev, alpha=0.62, label="k_src->AND(i)")
ax.axhline(0.0, color="#666666", lw=0.9)
ax.set_title("Same bins: local KL contributions")
ax.set_ylabel("Local KL value")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
ax.grid(alpha=0.20)
ax.legend(frameon=False, fontsize=8)
fig.suptitle(
    "From shared-bin masses to local KL spikes (real AND vs source_1 data)",
    y=0.995,
)
fig.tight_layout(rect=(0, 0, 1, 0.95))
save_and_show(fig, "plot_32b_shared_bin_local_kl_walkthrough.png")


### Intuition Booster for Plot 32 (small discrete example)

Before the KDE-based curves, here is a hand-checkable 6-bin example.

- `P_AND` is the reference distribution across bins.
- `Q_close` is a mild mismatch.
- `Q_tail_miss` under-covers high-`d_T` bins where `P_AND` still has mass.

For unit-width bins (`Δx=1`), local KL terms are:
$$
k_{AND	o src}(i)=p_i\log\dfrac{p_i}{q_i}, \quad
k_{src	o AND}(i)=q_i\log\dfrac{q_i}{p_i}
$$
So each bar is a per-bin contribution to total KL.


In [ ]:
toy_bins = np.arange(6)

# Hand-crafted PMFs on 6 bins (sum to 1)
p_and_toy = np.array([0.18, 0.24, 0.26, 0.18, 0.10, 0.04], dtype=float)
q_close = np.array([0.16, 0.23, 0.25, 0.20, 0.11, 0.05], dtype=float)
q_tail_miss = np.array([0.22, 0.28, 0.30, 0.17, 0.025, 0.005], dtype=float)

# Normalize defensively
p_and_toy = p_and_toy / p_and_toy.sum()
q_close = q_close / q_close.sum()
q_tail_miss = q_tail_miss / q_tail_miss.sum()


def local_terms_discrete(p, q, eps=1e-12):
    p_ = np.clip(p, eps, None)
    q_ = np.clip(q, eps, None)
    p_ = p_ / p_.sum()
    q_ = q_ / q_.sum()
    fwd = p_ * np.log(p_ / q_)
    rev = q_ * np.log(q_ / p_)
    return fwd, rev, fwd + rev


fwd_close, rev_close, j_close = local_terms_discrete(p_and_toy, q_close)
fwd_tail, rev_tail, j_tail = local_terms_discrete(p_and_toy, q_tail_miss)

# One explicit numeric example (single bin)
bin_idx = 4
p_i = p_and_toy[bin_idx]
q_i = q_tail_miss[bin_idx]
manual_term = p_i * np.log(p_i / q_i)

print("Single-bin worked example (tail-miss scenario):")
print(f"  bin={bin_idx}, p_i={p_i:.3f}, q_i={q_i:.3f}")
print(f"  k_AND->src(bin) = p_i * log(p_i/q_i) = {manual_term:.6f}")

toy_df = pd.DataFrame(
    {
        "bin": toy_bins,
        "P_AND": p_and_toy,
        "Q_close": q_close,
        "Q_tail_miss": q_tail_miss,
        "k_fwd_close": fwd_close,
        "k_rev_close": rev_close,
        "k_fwd_tail_miss": fwd_tail,
        "k_rev_tail_miss": rev_tail,
    }
)

print("Per-bin contributions:")
print(toy_df.round(5).to_string(index=False))

print("Totals:")
print(f"  KL(P_AND||Q_close)      = {fwd_close.sum():.5f}")
print(f"  KL(Q_close||P_AND)      = {rev_close.sum():.5f}")
print(f"  KL(P_AND||Q_tail_miss)  = {fwd_tail.sum():.5f}")
print(f"  KL(Q_tail_miss||P_AND)  = {rev_tail.sum():.5f}")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13.8, 8.6), sharex=True)

w = 0.34
x = toy_bins.astype(float)

# Top row: mass profiles
ax = axes[0, 0]
ax.bar(x - w/2, p_and_toy, width=w, color="#1f77b4", alpha=0.85, label="P_AND")
ax.bar(x + w/2, q_close, width=w, color="#2ca02c", alpha=0.70, label="Q_close")
ax.set_title("Mass profile: close mismatch")
ax.set_ylabel("Probability mass")
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.20)

ax = axes[0, 1]
ax.bar(x - w/2, p_and_toy, width=w, color="#1f77b4", alpha=0.85, label="P_AND")
ax.bar(x + w/2, q_tail_miss, width=w, color="#d62728", alpha=0.72, label="Q_tail_miss")
ax.set_title("Mass profile: tail under-coverage")
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.20)

# Bottom-left: forward local terms
ax = axes[1, 0]
colors_close = ["#2ca02c" if v >= 0 else "#666666" for v in fwd_close]
colors_tail = ["#d62728" if v >= 0 else "#666666" for v in fwd_tail]
ax.bar(x - w/2, fwd_close, width=w, color=colors_close, alpha=0.80, label=f"Q_close  (sum={fwd_close.sum():.3f})")
ax.bar(x + w/2, fwd_tail, width=w, color=colors_tail, alpha=0.80, label=f"Q_tail_miss  (sum={fwd_tail.sum():.3f})")
ax.axhline(0.0, color="#666666", lw=0.9)
ax.set_title(r"Local forward terms  $k_{AND	o src}(i)=p_i\log(p_i/q_i)$")
ax.set_ylabel("Per-bin KL contribution")
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.20)

# Bottom-right: reverse local terms
ax = axes[1, 1]
colors_close_r = ["#2ca02c" if v >= 0 else "#666666" for v in rev_close]
colors_tail_r = ["#d62728" if v >= 0 else "#666666" for v in rev_tail]
ax.bar(x - w/2, rev_close, width=w, color=colors_close_r, alpha=0.80, label=f"Q_close  (sum={rev_close.sum():.3f})")
ax.bar(x + w/2, rev_tail, width=w, color=colors_tail_r, alpha=0.80, label=f"Q_tail_miss  (sum={rev_tail.sum():.3f})")
ax.axhline(0.0, color="#666666", lw=0.9)
ax.set_title(r"Local reverse terms  $k_{src	o AND}(i)=q_i\log(q_i/p_i)$")
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.20)

for ax in axes[1, :]:
    ax.set_xlabel("d_T bin index")

fig.suptitle("Why Plot 32 matters: local KL spikes reveal where mismatch comes from", y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.97))
save_and_show(fig, "plot_32a_toy_local_kl_intuition.png")


### Reading this intuition block

- A high positive forward bar means: **AND has meaningful mass, source has too little mass there**.
- A high positive reverse bar means: **source places mass where AND has relatively little**.
- Negative bars are allowed locally; KL non-negativity only holds after summing across all bins.
- In the tail-miss case, a few bins dominate total KL, matching the Plot 32 intuition that local spikes identify the regime driving divergence.


In [ ]:
fig, axes = plt.subplots(len(sources), 2, figsize=(13.8, 7.2), sharex=True, squeeze=False)

for ridx, src in enumerate(sources):
    p_src = p_sources[src]
    src_label = dist_meta[src]["label"]
    src_color = dist_meta[src]["color"]

    k_fwd_y, kl_fwd = local_kl_density(p_and, p_src, dx)
    k_rev_y, kl_rev = local_kl_density(p_src, p_and, dx)

    ax0, ax1 = axes[ridx]
    plot_signed_fill(ax0, x_grid, k_fwd_y, src_color)
    overlay_kdes(ax0, x_grid, p_and, p_src, dx, and_color="#1f77b4", src_color=src_color)
    ax0.set_title(f"{src_label}\n$k_{{AND\to p^*}}(x)$, integral={kl_fwd:.3f}", fontsize=10)
    ax0.set_ylabel("Local KL density")

    plot_signed_fill(ax1, x_grid, k_rev_y, src_color)
    overlay_kdes(ax1, x_grid, p_and, p_src, dx, and_color="#1f77b4", src_color=src_color)
    ax1.set_title(f"{src_label}\n$k_{{p^*\to AND}}(x)$, integral={kl_rev:.3f}", fontsize=10)

    for ax in (ax0, ax1):
        ax.grid(alpha=0.20)

axes[-1][0].set_xlabel(r"$d_T$")
axes[-1][1].set_xlabel(r"$d_T$")
fig.suptitle("Plot 32 — Local signed KL contribution densities (pointwise scaffold 4/5)", y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.96))

save_and_show(fig, "plot_32_pointwise_scaffold_local_kl_terms.png")


### How to read Plot 32

- Positive regions add to KL; negative regions subtract locally.
- Forward and reverse directions can emphasize different \(d_T\) regions, showing KL asymmetry.
- The integral over each curve is the scalar KL value shown in the title.


## 8) Plot 33: cumulative KL area


In [ ]:
fig, axes = plt.subplots(len(sources), 1, figsize=(11.5, 6.2), sharex=True, squeeze=False)

for ridx, src in enumerate(sources):
    ax = axes[ridx][0]
    p_src = p_sources[src]
    src_label = dist_meta[src]["label"]
    src_color = dist_meta[src]["color"]

    local_fwd = p_and * np.log(p_and / p_src)
    local_rev = p_src * np.log(p_src / p_and)

    csum_fwd = np.cumsum(local_fwd)
    csum_rev = np.cumsum(local_rev)

    ax.plot(x_grid, csum_fwd, color=src_color, lw=2.1, label=r"running $KL(P_{AND}||P_{p^*})$")
    ax.plot(x_grid, csum_rev, color=src_color, lw=2.0, ls="--", label=r"running $KL(P_{p^*}||P_{AND})$")

    ax.axhline(csum_fwd[-1], color=src_color, lw=0.9, alpha=0.35)
    ax.axhline(csum_rev[-1], color=src_color, lw=0.9, alpha=0.35, ls="--")

    ax.text(x_grid[-1], csum_fwd[-1], f"  {csum_fwd[-1]:.3f}", va="center", color=src_color, fontsize=9)
    ax.text(x_grid[-1], csum_rev[-1], f"  {csum_rev[-1]:.3f}", va="center", color=src_color, fontsize=9)

    ax.set_ylabel("Running KL")
    ax.set_title(f"{src_label}: cumulative area up to each $d_T$", fontsize=10)
    ax.grid(alpha=0.20)
    ax.legend(loc="upper left", frameon=False, fontsize=9)

axes[-1][0].set_xlabel(r"$d_T$")
fig.suptitle("Plot 33 — Running KL integrals (pointwise scaffold 5/5)", y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.96))

save_and_show(fig, "plot_33_pointwise_scaffold_cumulative_kl.png")


### How to read Plot 33

- Each curve accumulates local contributions from left to right over \(d_T\).
- The endpoint equals the total KL divergence for that direction.
- Steep increases highlight \(d_T\) ranges where mismatch contributes most strongly.


## 9) Final Figure-7-style panel: pointwise KL / reverse-KL / Jeffreys


In [ ]:
fig, axes = plt.subplots(len(sources), 3, figsize=(16.0, 7.6), sharex=True, squeeze=False)

col_titles = [
    r"Forward KL: $D_{KL}(P_{AND}||P_{p^*})$",
    r"Reverse KL: $D_{KL}(P_{p^*}||P_{AND})$",
    r"Jeffreys: $J(P_{AND}, P_{p^*})$",
]

for cidx, title in enumerate(col_titles):
    axes[0][cidx].set_title(title, fontsize=11, fontweight="bold")

for ridx, src in enumerate(sources):
    p_src = p_sources[src]
    src_label = dist_meta[src]["label"]
    src_color = dist_meta[src]["color"]

    k_fwd_y, kl_fwd = local_kl_density(p_and, p_src, dx)
    k_rev_y, kl_rev = local_kl_density(p_src, p_and, dx)
    j_y, j_val = jeffreys_density(p_and, p_src, dx)

    y_terms = [k_fwd_y, k_rev_y, j_y]
    integrals = [kl_fwd, kl_rev, j_val]

    for cidx in range(3):
        ax = axes[ridx][cidx]
        y = y_terms[cidx]

        if cidx < 2:
            plot_signed_fill(ax, x_grid, y, src_color)
        else:
            ax.fill_between(x_grid, 0.0, y, color=src_color, alpha=0.32, linewidth=0)
            ax.plot(x_grid, y, color=src_color, lw=2.0)
            ax.axhline(0.0, color="#666666", lw=0.9, alpha=0.8)

        overlay_kdes(ax, x_grid, p_and, p_src, dx, and_color="#1f77b4", src_color=src_color)
        ax.grid(alpha=0.20)
        ax.text(
            0.02,
            0.93,
            f"Integral = {integrals[cidx]:.3f}",
            transform=ax.transAxes,
            fontsize=9,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.7),
        )

        if cidx == 0:
            ax.set_ylabel(f"{src_label}\nContribution density")

for cidx in range(3):
    axes[-1][cidx].set_xlabel(r"$d_T$")

fig.suptitle(
    "Figure-7-style pointwise divergence contributions (toy example)",
    fontsize=12,
    y=0.995,
)
fig.tight_layout(rect=(0, 0, 1, 0.96))

save_and_show(fig, "plot_24_pointwise_kl_jeffreys_figure7_style.png")


### How to interpret the final panel

- **Rows** compare a source that is closer vs farther from AND.
- **Forward KL column** emphasizes where AND-supported mass is under-covered by the source.
- **Reverse KL column** emphasizes where source mass appears in regions less supported by AND.
- **Jeffreys column** summarizes two-sided shape mismatch and is symmetric.
- The farther source typically shows larger integrated areas and stronger tail-driven mismatch.


## Key takeaways

- The scaffold converts raw 2D intuition into a 1D distance-distribution view suitable for divergence analysis.
- Plots 29→33 progressively reveal how local density mismatch becomes global KL/Jeffreys values.
- The final Figure-7-style panel is best read as **where** mismatch happens on \(d_T\), not just how much total mismatch exists.
